### Imports

In [18]:
import pandas as pd
import numpy as np
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
nlp.Defaults.stop_words -= {"not", "no", "n't", "never"}
for word in {"not", "no", "n't", "never"}:
    nlp.vocab[word].is_stop = False

### Preprocessing:

In [19]:
reviews_clean_df = pd.read_csv('../data/processed/reviews_clean.csv', index_col=0)

In [20]:
processed_df = reviews_clean_df[reviews_clean_df['Score'] != 3].copy()

In [21]:
processed_df["Sentiment"] = processed_df["Score"].isin([4, 5]).astype("int64")

In [22]:
processed_df["Sentiment"].value_counts()

Sentiment
1    443542
0     81966
Name: count, dtype: int64

In [23]:
processed_df["Text"] = processed_df["Text"].str.replace(r"<br\s*/?>", " ", regex=True)

In [24]:
row_lengths = processed_df['Text'].str.split().str.len()
processed_df.loc[row_lengths.idxmax(), 'Text']

'*********************************************************  UPDATE:  READ THE UPDATE BELOW FIRST. Thanks. (June 15 and August 3, 2012) *********************************************************   I WAS, BUT NO MORE (actively anyway). We have four cats: two ("normals") who eat about anything, one with kidney disease (CRF/CKD) and accompanying reduced appetite, and one that has picky (and weird) tastes.  In the process of trying to find an appetite stimulant for the mom-cat (with CRF/CKD) I tried out several products on her and the others (the kids) as well. Here are the results of this less than scientific (or definitive) survey:    Drs.Foster&Smith Shrimp Snappies Treats (ground to powder)-----------ALL REFUSED.   Freeze-dried powders (Prowl and Ziwi Peak)----------------------------NO EFFECT. All but one (Ms. Picky) ate them.   Seafood juices (tuna, oyster, sardine, etc.)-----------------------------NO EFFECT. All but (same) one ate them.   Dry (ground to powder) cat food (Halo, EVO, B

In [25]:
processed_df.loc[np.random.choice(processed_df.index), 'Text']

"I was desperate when Lea & Perrins BBQ Sauce was discontinued - but never fear! This BBQ sauce is as good or better! You won't go wrong if you try it. It's a little sweet, but not overbearing. Give it a shot!"

In [26]:
processed_df["Text"] = processed_df["Text"].str.replace(r"<[^>]+>", " ", regex=True)

In [27]:
processed_df.loc[np.random.choice(processed_df.index), 'Text']

"Once again, a customer for life with Amazon's subscribe and save program. This coffee is awesome! With the price being a little over $7.00 per 12 oz bag versus $11.00 at he store, this is a no brainer. It's organic, fair trade and an extra bonus for all my composting friends, the bag is bio-degradible! Absolutely nothing goes to waste or landfill. I've given the beans to several co-workers and the reaction is all the same, that's really good coffee! If you are a coffee lover give Larry's Beans a try. They are in the business for all the right reasons and it shows."

In [28]:
processed_df["Text"] = processed_df["Text"].str.lower()

In [29]:
# Removed: punctuation stripping here broke contractions (e.g. "didn't" -> "didn", "t")
# before spaCy could tokenize them correctly. Punctuation is instead filtered out
# later via token.is_alpha in the lemmatization step.
# processed_df["Text"] = processed_df["Text"].str.replace(r"[^\w\s]", " ", regex=True)

In [30]:
processed_df["Text"] = processed_df["Text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [31]:
processed_df.loc[np.random.choice(processed_df.index), 'Text']

"this salt is highly recommended by dr. larry wilson for it's high selenium content. i was surprised that it also very tasty and i find myself craving it. it goes well on anything you are already salting. i have my whole family switching."

In [32]:
def preprocess_text(doc):
    return " ".join([token.lemma_ for token in doc if not token.is_stop and token.is_alpha])

In [33]:
processed_df["Text_clean"] = [preprocess_text(doc) for doc in nlp.pipe(processed_df["Text"], batch_size=1000, n_process=4)]

In [34]:
processed_df[["Text", "Text_clean"]].sample(5, random_state=1)

,Text,Text_clean
Id,,
71588,perfect for cooking and they taste great even ...,perfect cook taste great snack sweet completel...
495112,i stumbled upon these cinnamon toast crunch tr...,stumble cinnamon toast crunch treat local walm...
99850,i was a little worried that i was wasting my m...,little worried waste money buy coffee price su...
135133,i love to make kasha stuffed cabbage rolls and...,love kasha stuff cabbage roll have trouble fin...
483425,i didn't start out with any health problems wh...,start health problem begin take bragg apple ci...


In [35]:
processed_df.to_csv('../data/processed/reviews_preprocessed.csv', index=True)

### Text preprocessing summary

Applied the following pipeline to clean review text:
- Stripped HTML tags (e.g. `<br />`, `<a href="...">`) using regex
- Lowercased text
- Tokenized, removed stop words, and lemmatized using spaCy (`en_core_web_sm`, with parser
  and NER disabled for performance on 500K+ rows). Punctuation was deliberately **not**
  stripped via regex beforehand — spaCy's tokenizer handles contractions (e.g. "didn't")
  correctly on its own, while manual punctuation removal broke them into meaningless
  fragments ("didn", "t"). The `token.is_alpha` filter in the lemmatization step removes
  punctuation as a side effect, after tokenization has already happened correctly.
- Negation words ("not", "no", "n't", "never") were explicitly excluded from spaCy's default
  stop word list. By default, "not" is treated as a stop word and removed — this is harmful
  for sentiment analysis, since dropping negation can flip the apparent meaning of a review
  (e.g. "did not have any results" losing "not" and reading as neutral/positive instead of negative).

Binary sentiment target was created from Score: reviews with Score 4-5 labeled as positive,
1-2 as negative, and Score 3 (neutral) dropped from the dataset.